In [1]:
## requirements library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle 
import streamlit as st 
import tensorboard as tb 


In [4]:
## Load the dataset 
df=pd.read_csv('Churn_Modelling.csv')
df.head()
print(df.shape)

(10000, 14)


In [8]:
## Remove the Irrevalent columns 
# like ID , Name, Row, Number 
df=df.drop(['RowNumber'],axis=1)
df=df.drop(['Surname'],axis=1)

df=df.drop(['CustomerId'],axis=1)

df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
## Encoding Categorical Variable
from sklearn.preprocessing import LabelEncoder
lebel_encoder = LabelEncoder()

In [10]:
df['Gender']=lebel_encoder.fit_transform(df['Gender'])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [11]:
## Then Encode the geographicay column so in this column we apply the onehotencoder 
from sklearn.preprocessing import OneHotEncoder
one_hot=OneHotEncoder()
geo_encoder = one_hot.fit_transform(df[['Geography']])
one_hot.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [16]:
## Convert to array 
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [17]:
geo_encoder_data= pd.DataFrame(geo_encoder.toarray(),columns = one_hot.get_feature_names_out(['Geography']))

In [18]:
geo_encoder_data.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [19]:
## Combine the one_hot encoded columns with original data 
df=pd.concat([df.drop('Geography',axis=1),geo_encoder_data],axis=1)
df.head()

KeyError: "['Geography'] not found in axis"

In [22]:
one_hot

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infrequent

In [23]:
## Save the encoder and scaler 
with open('level_encoder_gender.pkl','wb')as file :
    pickle.dump(lebel_encoder,file)
with open('geo_encoder.pkl','wb') as file:
    pickle.dump(one_hot,file)


## Divide the dataset into dependent and independent feature

In [25]:
from sklearn.model_selection import train_test_split 
X=df.drop('Exited',axis=1)
y=df['Exited']
print(X.shape)
print(y.shape)

(10000, 12)
(10000,)


In [26]:
## Split the data into train test 
X_train,X_test,y_train,y_test  = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 12)
(2000, 12)
(8000,)
(2000,)


In [27]:
## Select the features 
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [28]:
X_train_scale=scaler.fit_transform(X_train)
X_test_scale=scaler.transform(X_test)


In [29]:
## Samve the Scaler Model 
with open('scaler.pkl','wb')as file:
    pickle.dump(scaler,file)

# ANN :- Artficial Neural Network 
## Implementation 

In [30]:
import tensorflow as tf 


In [31]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime 

In [32]:
X_train.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
9254,686,1,32,6,0.00,2,1,1,179093.26,1.0,0.0,0.0
1561,632,1,42,4,119624.60,2,1,1,195978.86,0.0,1.0,0.0
1670,559,1,24,3,114739.92,1,1,0,85891.02,0.0,0.0,1.0
6087,561,0,27,9,135637.00,1,1,0,153080.40,1.0,0.0,0.0
6669,517,1,56,9,142147.32,1,0,0,39488.04,1.0,0.0,0.0


In [33]:
## Build Our ANN Model 
model = Sequential(
    [
        Dense(64,activation='relu',input_shape=(X_train_scale.shape[1],)),  # HL1
        Dense(32,activation='relu'), # HL2 
        Dense(1,activation='sigmoid') # Output Layer 
    ]
)

C:\Users\Madhav Yadav\OneDrive\Desktop\desktop1\Datascience_bootcamp\Deep_Learning\ANN\Churn_prediction\ann_venv\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │             832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [36]:
## Compile the model  
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [37]:
## Setup the tensor board 
log_dir='logs/fit' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [38]:
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [39]:
## Set up Early Stopping 
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=20,restore_best_weights=True)


In [40]:
### Training The Model 
history = model.fit(X_train_scale,y_train,validation_data=(X_test_scale,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8375 - loss: 0.3948 - val_accuracy: 0.8560 - val_loss: 0.3536
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8529 - loss: 0.3560 - val_accuracy: 0.8630 - val_loss: 0.3398
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8585 - loss: 0.3480 - val_accuracy: 0.8640 - val_loss: 0.3356
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8608 - loss: 0.3444 - val_accuracy: 0.8575 - val_loss: 0.3391
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8619 - loss: 0.3413 - val_accuracy: 0.8630 - val_loss: 0.3343
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8636 - loss: 0.3382 - val_accuracy: 0.8580 - val_loss: 0.3383
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8643 - loss: 0.3348 - val_accuracy: 0.8585 - val_loss: 0.3380
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8674 - loss: 0.3311 - val_acc

In [41]:
model.save('model.h5')

In [54]:
import sys
print(sys.executable)

C:\Users\Madhav Yadav\OneDrive\Desktop\desktop1\Datascience_bootcamp\Deep_Learning\ANN\Churn_prediction\ann_venv\Scripts\python.exe


In [59]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [61]:
%tensorboard --logdir  logs/fit.84a81934-016e-4b92-9f98-eac6517711e3